# Notebook 01 — Ingest Papers from OpenAlex

This notebook downloads academic papers from the OpenAlex API and saves a normalized CSV.

**What it does:**
1. Loads keywords and settings from `config/settings.yaml`
2. Searches OpenAlex for papers matching the synthetic biology keywords
3. Normalizes the results to the shared schema
4. Tags carbon-capture papers with `case_study_flag = True`
5. Saves `data/processed/papers.csv`

**Before running:**
- Set `OPENALEX_EMAIL` in your `.env` file (optional but recommended)
- Run from the repository root directory

In [ ]:
import sys
sys.path.insert(0, '..')  # add repo root to path

from pathlib import Path
from src.utils.config import load_config
from src.ingest import openalex, normalize

cfg = load_config()
print('Config loaded.')
print('Seed keywords:', cfg['corpus']['seed_keywords'])

In [ ]:
# --- 1. Fetch papers from OpenAlex ---

raw_works = []

for work in openalex.search_papers(
    keywords=cfg['corpus']['seed_keywords'],
    year_min=cfg['corpus']['year_min'],
    year_max=cfg['corpus']['year_max'],
    max_results=cfg['corpus']['openalex_max_results'],
):
    raw_works.append(openalex.extract_fields(work))

print(f'Downloaded {len(raw_works)} works from OpenAlex.')

In [ ]:
# --- 2. Normalize to shared schema ---

papers_df = normalize.normalize_papers(
    raw_records=raw_works,
    carbon_keywords=cfg['corpus']['carbon_capture_keywords'],
)

print(f'Normalized: {len(papers_df)} papers')
print(f'Carbon capture: {papers_df["case_study_flag"].sum()} papers tagged')
papers_df.head()

In [ ]:
# --- 3. Save to CSV ---

output_path = Path('../data/processed/papers.csv')
output_path.parent.mkdir(parents=True, exist_ok=True)
papers_df.to_csv(output_path, index=False)
print(f'Saved to {output_path}')

In [ ]:
# --- 4. Quick inspection ---

print('Year range:', papers_df['year'].min(), '—', papers_df['year'].max())
print('\nTop countries:')
print(papers_df['country'].value_counts().head(10))
print('\nTop cities:')
print(papers_df['city'].value_counts().head(10))